# Обучение

In [1]:
import json
import sys
import time
from pathlib import Path

import numpy as np
import polars as pl
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    top_k_accuracy_score,
)

from catboost import CatBoostClassifier, Pool

from genre_ensemble import GenreStackingClassifier

sys.path.insert(0, str(Path.cwd()))

DATA_PATH = Path("../../spotify_data.csv")
DATA_DIR = Path("../data")
SPLITS_DIR = DATA_DIR / "splits"
MODELS_DIR = Path("../models")
ENSEMBLE_DIR = MODELS_DIR / "ensemble"
for d in (DATA_DIR, SPLITS_DIR, MODELS_DIR, ENSEMBLE_DIR):
    d.mkdir(parents=True, exist_ok=True)

SEED = 123
FEATURES = [
    "danceability",
    "energy",
    "key",
    "loudness",
    "mode",
    "speechiness",
    "acousticness",
    "instrumentalness",
    "liveness",
    "valence",
    "tempo",
    "duration_ms",
    "time_signature",
]
EXTRA_COLS = ["popularity", "year", "track_id", "artist_name", "track_name"]

In [2]:
SINGLE_CB_ITERS = 600
LGBM_ITERS = 300
CB_ITERS = 400

In [3]:
needed = {
    p.name
    for p in [
        SPLITS_DIR / "train.parquet",
        SPLITS_DIR / "val.parquet",
        SPLITS_DIR / "test.parquet",
    ]
}
existing = {p.name for p in SPLITS_DIR.glob("*.parquet")}

if not needed.issubset(existing):
    from sklearn.model_selection import StratifiedShuffleSplit

    print("building splits from", DATA_PATH)
    raw = pl.read_csv(DATA_PATH).drop("")
    X = raw.select(FEATURES + EXTRA_COLS).to_pandas()
    y = raw["genre"].to_numpy()
    sa = StratifiedShuffleSplit(n_splits=1, test_size=0.30, random_state=SEED)
    train_idx, hold_idx = next(sa.split(X, y))
    sb = StratifiedShuffleSplit(n_splits=1, test_size=0.50, random_state=SEED)
    val_idx, test_idx = next(sb.split(X.iloc[hold_idx], y[hold_idx]))
    Xh, yh = X.iloc[hold_idx], y[hold_idx]
    for name, sub_X, sub_y in [
        ("train", X.iloc[train_idx], y[train_idx]),
        ("val", Xh.iloc[val_idx], yh[val_idx]),
        ("test", Xh.iloc[test_idx], yh[test_idx]),
    ]:
        pl.from_pandas(sub_X).with_columns(pl.Series("genre", sub_y)).write_parquet(
            SPLITS_DIR / f"{name}.parquet"
        )
        print(f"  {name}: {len(sub_X)} rows")
else:
    print("splits present:", sorted(existing))

splits present: ['test.parquet', 'train.parquet', 'val.parquet']


In [4]:
train_df = pl.read_parquet(SPLITS_DIR / "train.parquet").to_pandas()
val_df = pl.read_parquet(SPLITS_DIR / "val.parquet").to_pandas()
test_df = pl.read_parquet(SPLITS_DIR / "test.parquet").to_pandas()

classes_path = MODELS_DIR / "label_classes.npy"
if classes_path.exists():
    classes = np.load(classes_path, allow_pickle=True).tolist()
else:
    classes = sorted(train_df["genre"].unique().tolist())
    np.save(classes_path, np.array(classes))
label_map = {n: i for i, n in enumerate(classes)}
n_classes = len(classes)


def encode(df):
    return df[FEATURES].copy(), np.array([label_map[g] for g in df["genre"].to_numpy()])


X_train, y_train = encode(train_df)
X_val, y_val = encode(val_df)
X_test, y_test = encode(test_df)
print(
    f"train={X_train.shape} val={X_val.shape} test={X_test.shape} classes={n_classes}"
)

train=(811834, 13) val=(173965, 13) test=(173965, 13) classes=82


In [5]:
single = CatBoostClassifier(
    iterations=SINGLE_CB_ITERS,
    depth=6,
    learning_rate=0.08,
    l2_leaf_reg=4.0,
    bagging_temperature=0.5,
    border_count=64,
    loss_function="MultiClass",
    eval_metric="TotalF1:average=Macro",
    auto_class_weights="Balanced",
    task_type="GPU",
    devices="0",
    random_seed=SEED,
    early_stopping_rounds=60,
    verbose=200,
)
t0 = time.time()
single.fit(
    Pool(X_train, label=y_train), eval_set=Pool(X_val, label=y_val), use_best_model=True
)
print(f"single catboost: {time.time() - t0}s  best_iter={single.get_best_iteration()}")
single.save_model(str(MODELS_DIR / "catboost_genre.cbm"))

0:	learn: 0.0473526	test: 0.0468142	best: 0.0468142 (0)	total: 189ms	remaining: 1m 53s
200:	learn: 0.2289593	test: 0.2173426	best: 0.2173426 (200)	total: 26.5s	remaining: 52.6s
400:	learn: 0.2513758	test: 0.2315057	best: 0.2315057 (400)	total: 49s	remaining: 24.3s
599:	learn: 0.2664477	test: 0.2379671	best: 0.2381195 (596)	total: 1m 9s	remaining: 0us
bestTest = 0.2381194555
bestIteration = 596
Shrink model to first 597 iterations.
single catboost: 71.47026348114014s  best_iter=596


In [6]:
ensemble = GenreStackingClassifier(
    features=FEATURES, class_names=classes, seed=SEED, gpu=True, lgbm_device="gpu"
)
ensemble.lgbm_params = {**ensemble._default_lgbm(), "n_estimators": LGBM_ITERS}
ensemble.catboost_params = {**ensemble._default_catboost(), "iterations": CB_ITERS}

t0 = time.time()
ensemble.fit(X_train, y_train, X_val, y_val)
print(f"ensemble: {time.time() - t0}s")
print("fit_seconds:", ensemble.fit_seconds_)
ensemble.save(ENSEMBLE_DIR)

[lgbm] fitting
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[266]	valid_0's multi_logloss: 2.74704
[lgbm] done in 1843.5s
[catboost] fitting
0:	learn: 0.0473526	test: 0.0468142	best: 0.0468142 (0)	total: 126ms	remaining: 50.3s
200:	learn: 0.2289593	test: 0.2173426	best: 0.2173426 (200)	total: 24.1s	remaining: 23.8s
399:	learn: 0.2513212	test: 0.2313033	best: 0.2313033 (399)	total: 48.9s	remaining: 0us
bestTest = 0.2313032822
bestIteration = 399
[catboost] done in 50.7s
[meta] fitting logistic on val probabilities
[meta] done in 13.2s
[calibrate] per-class log-prior shift
[calibrate] macro_f1 after pass: 0.2682
[calibrate] macro_f1 after pass: 0.2696
ensemble: 2027.294703245163s
fit_seconds: {'lgbm': 1843.4620339870453, 'catboost': 50.66534948348999, 'meta': 13.205483675003052, 'calibrate': 119.62162017822266}


In [7]:
def evaluate(proba, y):
    pred = np.argmax(proba, axis=1)
    return {
        "top1_acc": float(accuracy_score(y, pred)),
        "top3_acc": float(
            top_k_accuracy_score(y, proba, k=3, labels=np.arange(n_classes))
        ),
        "top5_acc": float(
            top_k_accuracy_score(y, proba, k=5, labels=np.arange(n_classes))
        ),
        "macro_f1": float(f1_score(y, pred, average="macro", zero_division=0)),
        "weighted_f1": float(f1_score(y, pred, average="weighted", zero_division=0)),
    }


single_proba_test = single.predict_proba(X_test)
ens_proba_test = ensemble.predict_proba(X_test, calibrated=True)

metrics = {
    "single_catboost": {
        "val": evaluate(single.predict_proba(X_val), y_val),
        "test": evaluate(single_proba_test, y_test),
    },
    "ensemble": {
        "val": evaluate(ensemble.predict_proba(X_val, calibrated=True), y_val),
        "test": evaluate(ens_proba_test, y_test),
    },
    "base_val": ensemble.val_metrics_,
    "fit_seconds": ensemble.fit_seconds_,
}
with open(MODELS_DIR / "metrics.json", "w") as f:
    json.dump(metrics, f, indent=2, default=float)

for name, m in [
    ("single_catboost", metrics["single_catboost"]["test"]),
    ("ensemble", metrics["ensemble"]["test"]),
]:
    print(f"\n{name}:")
    for k, v in m.items():
        print(f"  {k} = {v}")


single_catboost:
  top1_acc = 0.2756129106429454
  top3_acc = 0.4701922800563332
  top5_acc = 0.5725404535395051
  macro_f1 = 0.23500280932833956
  weighted_f1 = 0.2558215103597046

ensemble:
  top1_acc = 0.2843100623688673
  top3_acc = 0.46537521915327795
  top5_acc = 0.561756675193286
  macro_f1 = 0.26475583868112207
  weighted_f1 = 0.28720459282429855


In [8]:
reloaded = GenreStackingClassifier.load(ENSEMBLE_DIR)
p1 = ensemble.predict_proba(X_test.iloc[:256])
p2 = reloaded.predict_proba(X_test.iloc[:256])
print(f"reload max_abs_diff = {float(np.max(np.abs(p1 - p2)))}")

reload max_abs_diff = 0.0
